# 03 Baseline results from the BanTH paper

CSE 4889 Machine Learning: Explainable Multi-Label Hate Speech Detection in Transliterated Bangla

We don't re-train the BanTH models. We take the published test results from the BanTH paper (Haider et al., Findings of NAACL 2025, Table 7) and save them as our baseline table. Our own model in the training notebook is compared against these numbers.

Source: https://aclanthology.org/2025.findings-naacl.403/ (numbers checked against arXiv 2410.13281v4, Table 7).

## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports, paths and helpers

Same paths and helper functions as the data notebook. Tables are saved as CSV and LaTeX, and each file is added to ASSETS.md.

In [2]:
import os
import json
from datetime import datetime
import pandas as pd

BASE = '/content/drive/MyDrive/ML_Project'
SEED = 42
NOTEBOOK_NAME = '03_baseline_table.ipynb'
TAB_DIR = os.path.join(BASE, 'results', 'tables')
REP_DIR = os.path.join(BASE, 'results', 'evaluation_reports')
ASSETS_PATH = os.path.join(BASE, 'results', 'ASSETS.md')
assert os.path.exists(os.path.join(REP_DIR, 'metrics_data.json')), 'Run 01_data_exploration.ipynb first.'

def save_table(df, name, float_format='%.2f'):
    df.to_csv(os.path.join(TAB_DIR, f'{name}.csv'), index=False)
    df.to_latex(os.path.join(TAB_DIR, f'{name}.tex'), index=False, escape=True,
                float_format=float_format, na_rep='--')
    print('Saved:', name, '(.csv and .tex)')

def register_asset(asset_id, file_name, what, notebook, section, caption):
    row = f'| {asset_id} | {file_name} | {what} | {notebook} | {section} | {caption} |'
    with open(ASSETS_PATH, encoding='utf-8') as f:
        lines = f.read().splitlines()
    lines = [l for l in lines if not l.startswith(f'| {asset_id} |')]
    lines.append(row)
    with open(ASSETS_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print('Registered', asset_id, '->', file_name)

## Published results (Table 7 of the BanTH paper)

All numbers are copied exactly as printed, in percent, on the official test split (3,735 comments).

How the paper measures them:
- Binary: macro-F1 and macro-accuracy (the average of the accuracy on hate and on non-hate, which is the same as balanced accuracy).
- Multi-label: macro-F1 over the categories, subset accuracy (all labels of a comment must be right), and Hamming loss (share of wrong labels, lower is better).

The fine-tuned models used learning rate 2e-5, Adam, max length 512, 5 epochs, batch size 32 and early stopping on validation loss. The TB models were first further pre-trained on transliterated Bangla text, and the GPT rows are prompting only, with no training.

In [3]:
COLS = ['Group', 'Model', 'Binary Macro-F1', 'Binary Macro-Acc.',
        'Multi-label Macro-F1', 'Subset Acc.', 'Hamming Loss']

PAPER_ROWS = [
    ('Fine-tuned (Bangla LM)', 'BanglaBERT', 76.50, 81.04, 20.82, 54.08, 7.26),
    ('Fine-tuned (Bangla LM)', 'BanglishBERT', 75.07, 80.62, 20.61, 52.74, 7.42),
    ('Fine-tuned (Bangla LM)', 'BanglaHateBERT', 70.92, 77.54, 11.34, 49.83, 7.97),
    ('Fine-tuned (Bangla LM)', 'VAC-BERT', 74.19, 79.76, 18.45, 52.56, 7.50),
    ('Fine-tuned (Indian LM)', 'MuRIL', 75.29, 80.83, 14.58, 53.98, 7.55),
    ('Fine-tuned (Indian LM)', 'IndicBERT', 74.51, 80.48, 13.39, 51.56, 7.88),
    ('Fine-tuned (Multilingual LM)', 'mBERT', 74.97, 80.37, 28.24, 52.19, 7.78),
    ('Fine-tuned (Multilingual LM)', 'XLM-R', 77.35, 81.37, 29.29, 53.28, 7.23),
    ('Fine-tuned (Character LM)', 'CharBERT', 76.61, 80.91, 19.66, 53.21, 7.44),
    ('Further pre-trained (TB LM)', 'TB-BERT', 76.27, 79.25, 30.17, 54.19, 7.18),
    ('Further pre-trained (TB LM)', 'TB-mBERT', 77.36, 82.57, 27.07, 54.71, 7.28),
    ('Further pre-trained (TB LM)', 'TB-XLM-R', 77.04, 81.29, 29.04, 52.86, 7.26),
    ('Further pre-trained (TB LM)', 'TB-BanglaBERT', 77.12, 81.61, 22.52, 53.97, 7.37),
    ('Further pre-trained (TB LM)', 'TB-BanglishBERT', 77.12, 81.39, 21.41, 52.79, 7.42),
    ('LLM prompting (Vanilla)', 'GPT 3.5', 61.44, 64.02, 24.12, 18.27, 19.77),
    ('LLM prompting (Vanilla)', 'GPT 4o', 70.05, 74.30, 36.07, 22.60, 16.91),
    ('LLM prompting (Vanilla)', 'GPT 3.5 + Few-shot', 61.85, 65.65, 23.86, 16.85, 20.36),
    ('LLM prompting (Vanilla)', 'GPT 4o + Few-shot', 68.77, 74.18, 39.53, 26.76, 14.16),
    ('LLM prompting (CoT)', 'GPT 3.5 + CoT', 61.87, 65.77, 25.61, 20.34, 19.28),
    ('LLM prompting (CoT)', 'GPT 4o + CoT', 69.87, 74.14, 35.97, 22.60, 16.64),
    ('LLM prompting (CoT)', 'GPT 3.5 + CoT + Few-shot', 63.30, 67.80, 28.35, 20.15, 19.15),
    ('LLM prompting (CoT)', 'GPT 4o + CoT + Few-shot', 69.50, 74.94, 36.49, 21.10, 17.36),
    ('LLM prompting (Explanation)', 'GPT 3.5 + Exp', 61.69, 65.16, 22.61, 17.33, 19.84),
    ('LLM prompting (Explanation)', 'GPT 4o + Exp', 69.91, 74.16, 32.58, 19.12, 17.94),
    ('LLM prompting (Explanation)', 'GPT 3.5 + Exp + Few-shot', 62.69, 66.60, 25.00, 18.36, 19.57),
    ('LLM prompting (Explanation)', 'GPT 4o + Exp + Few-shot', 70.70, 75.15, 35.61, 21.33, 17.15),
    ('LLM prompting (HARE)', 'GPT 3.5 + HARE', 62.64, 66.77, 25.99, 20.90, 18.94),
    ('LLM prompting (HARE)', 'GPT 4o + HARE', 69.71, 74.75, 36.67, 20.80, 17.10),
    ('LLM prompting (HARE)', 'GPT 3.5 + HARE + Few-shot', 62.20, 66.72, 24.68, 20.06, 19.18),
    ('LLM prompting (HARE)', 'GPT 4o + HARE + Few-shot', 69.12, 74.75, 34.06, 21.59, 17.37),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [BAN]', 69.63, 72.70, 35.72, 23.28, 16.73),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [ENG]', 69.38, 72.96, 34.28, 20.72, 16.91),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [BAN] + CoT', 69.01, 72.66, 36.22, 21.35, 17.36),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [ENG] + CoT', 69.74, 74.87, 35.94, 21.19, 16.91),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [BAN] + Exp', 70.34, 73.65, 36.20, 21.54, 16.46),
    ('LLM prompting (Translation)', 'GPT 4o + Translation [ENG] + Exp', 70.19, 73.84, 36.25, 23.92, 16.23),
]

full = pd.DataFrame(PAPER_ROWS, columns=COLS)
print(len(full), 'rows')
full

36 rows


,Group,Model,Binary Macro-F1,Binary Macro-Acc.,Multi-label Macro-F1,Subset Acc.,Hamming Loss
0,Fine-tuned (Bangla LM),BanglaBERT,76.50,81.04,20.82,54.08,7.26
1,Fine-tuned (Bangla LM),BanglishBERT,75.07,80.62,20.61,52.74,7.42
2,Fine-tuned (Bangla LM),BanglaHateBERT,70.92,77.54,11.34,49.83,7.97
3,Fine-tuned (Bangla LM),VAC-BERT,74.19,79.76,18.45,52.56,7.50
4,Fine-tuned (Indian LM),MuRIL,75.29,80.83,14.58,53.98,7.55
5,Fine-tuned (Indian LM),IndicBERT,74.51,80.48,13.39,51.56,7.88
6,Fine-tuned (Multilingual LM),mBERT,74.97,80.37,28.24,52.19,7.78
7,Fine-tuned (Multilingual LM),XLM-R,77.35,81.37,29.29,53.28,7.23
8,Fine-tuned (Character LM),CharBERT,76.61,80.91,19.66,53.21,7.44
9,Further pre-trained (TB LM),TB-BERT,76.27,79.25,30.17,54.19,7.18


## Best result for each metric

We let the code find the best model per metric, so the claims in the report (for example which model is best at binary detection) come from the table and not from reading it by eye. For Hamming loss, lower is better.

In [4]:
metric_cols = COLS[2:]
fine_tuned = full[~full['Group'].str.startswith('LLM')]

best = {}
for scope, df in [('all_methods', full), ('fine_tuned_only', fine_tuned)]:
    best[scope] = {}
    for col in metric_cols:
        idx = df[col].idxmin() if col == 'Hamming Loss' else df[col].idxmax()
        best[scope][col] = {'model': df.loc[idx, 'Model'], 'value': float(df.loc[idx, col])}
        print(f'{scope:<16} {col:<22} {df.loc[idx, "Model"]:<22} {df.loc[idx, col]:.2f}')

all_methods      Binary Macro-F1        TB-mBERT               77.36
all_methods      Binary Macro-Acc.      TB-mBERT               82.57
all_methods      Multi-label Macro-F1   GPT 4o + Few-shot      39.53
all_methods      Subset Acc.            TB-mBERT               54.71
all_methods      Hamming Loss           TB-BERT                7.18
fine_tuned_only  Binary Macro-F1        TB-mBERT               77.36
fine_tuned_only  Binary Macro-Acc.      TB-mBERT               82.57
fine_tuned_only  Multi-label Macro-F1   TB-BERT                30.17
fine_tuned_only  Subset Acc.            TB-mBERT               54.71
fine_tuned_only  Hamming Loss           TB-BERT                7.18


## Save the full table (T07)

All 36 rows are saved. This is too long for the paper itself, but it is useful as a reference and for the appendix.

In [5]:
save_table(full, 'tab_07_banth_published_results_full')
register_asset('T07', 'tables/tab_07_banth_published_results_full.csv (+ .tex)',
               'All 36 published BanTH test results (fine-tuned encoders and GPT prompting), copied from the BanTH paper Table 7',
               NOTEBOOK_NAME, 'Appendix / reference',
               'TABLE X. Published results on the BanTH test split, reproduced from Haider et al. All values are percentages.')

Saved: tab_07_banth_published_results_full (.csv and .tex)
Registered T07 -> tables/tab_07_banth_published_results_full.csv (+ .tex)


## Short baseline table for the report (T08)

For the paper we keep a few rows:
- BanglaBERT, BanglishBERT, mBERT and XLM-R, the normal fine-tuned models that are closest to what we train,
- TB-mBERT and TB-BERT, the best fine-tuned models for binary and multi-label,
- GPT 4o zero-shot, and GPT 4o + Few-shot, which has the best multi-label macro-F1 overall.

Our own model's row is added in the training notebook, from its saved test results.

In [7]:
SHORT_MODELS = ['BanglaBERT', 'BanglishBERT', 'mBERT', 'XLM-R', 'TB-BERT', 'TB-mBERT',
                'GPT 4o', 'GPT 4o + Few-shot']
short = full[full['Model'].isin(SHORT_MODELS)].copy()
short.insert(0, 'Source', 'BanTH paper')
short = short.drop(columns='Group')
display(short)

save_table(short, 'tab_08_baseline_comparison')
register_asset('T08', 'tables/tab_08_baseline_comparison.csv (+ .tex)',
               'Selected published BanTH baselines; the training notebook adds our model row',
               NOTEBOOK_NAME, 'V. Results (comparison with baseline)',
               'TABLE X. Comparison with published baselines on the BanTH test split. Baseline values are taken from Haider et al. All values are percentages; lower Hamming loss is better.')

,Source,Model,Binary Macro-F1,Binary Macro-Acc.,Multi-label Macro-F1,Subset Acc.,Hamming Loss
0,BanTH paper,BanglaBERT,76.50,81.04,20.82,54.08,7.26
1,BanTH paper,BanglishBERT,75.07,80.62,20.61,52.74,7.42
6,BanTH paper,mBERT,74.97,80.37,28.24,52.19,7.78
7,BanTH paper,XLM-R,77.35,81.37,29.29,53.28,7.23
9,BanTH paper,TB-BERT,76.27,79.25,30.17,54.19,7.18
10,BanTH paper,TB-mBERT,77.36,82.57,27.07,54.71,7.28
15,BanTH paper,GPT 4o,70.05,74.30,36.07,22.60,16.91
17,BanTH paper,GPT 4o + Few-shot,68.77,74.18,39.53,26.76,14.16


Saved: tab_08_baseline_comparison (.csv and .tex)
Registered T08 -> tables/tab_08_baseline_comparison.csv (+ .tex)


,Source,Model,Binary Macro-F1,Binary Macro-Acc.,Multi-label Macro-F1,Subset Acc.,Hamming Loss
0,BanTH paper,BanglaBERT,76.50,81.04,20.82,54.08,7.26
1,BanTH paper,BanglishBERT,75.07,80.62,20.61,52.74,7.42
6,BanTH paper,mBERT,74.97,80.37,28.24,52.19,7.78
7,BanTH paper,XLM-R,77.35,81.37,29.29,53.28,7.23
9,BanTH paper,TB-BERT,76.27,79.25,30.17,54.19,7.18
10,BanTH paper,TB-mBERT,77.36,82.57,27.07,54.71,7.28
15,BanTH paper,GPT 4o,70.05,74.30,36.07,22.60,16.91
17,BanTH paper,GPT 4o + Few-shot,68.77,74.18,39.53,26.76,14.16


Saved: tab_08_baseline_comparison (.csv and .tex)
Registered T08 -> tables/tab_08_baseline_comparison.csv (+ .tex)


## Paper counts vs the HuggingFace data we use (T09)

While checking the paper we noticed that its category counts (Table 1) are not the same as in the public HuggingFace dataset we loaded. The paper describes 7 categories, while the HuggingFace version has 8 (it adds Misc), and several counts differ (for example Body Shaming is 68 in the paper's train split and 92 in ours). The split sizes are the same.

So the baseline numbers may have been measured on a slightly different label version. We save this comparison so the report can say so honestly. Our counts are read from metrics_data.json from the data notebook.

In [8]:
# Paper Table 1: category -> (train, val, test, total). The paper has no Misc row.
PAPER_TABLE1 = {
    'Political': (958, 120, 122, 1200),
    'Religious': (111, 14, 14, 139),
    'Gender': (154, 19, 20, 193),
    'Personal Offense': (5572, 696, 696, 6964),
    'Abusive/Violence': (2312, 287, 289, 2888),
    'Origin': (171, 21, 22, 214),
    'Body Shaming': (68, 8, 10, 86),
}
PAPER_HATE_TOTAL = 10282  # paper Table 4, total hate samples

with open(os.path.join(REP_DIR, 'metrics_data.json'), encoding='utf-8') as f:
    data_metrics = json.load(f)
hf = data_metrics['category_counts']

rows = []
for cat in hf['train']:
    p = PAPER_TABLE1.get(cat)
    hf_total = hf['train'][cat] + hf['validation'][cat] + hf['test'][cat]
    rows.append({'Category': cat,
                 'Paper train': p[0] if p else None, 'HF train': hf['train'][cat],
                 'Paper test': p[2] if p else None, 'HF test': hf['test'][cat],
                 'Paper total': p[3] if p else None, 'HF total': hf_total})
hf_hate_total = sum(data_metrics['hate_counts'].values())
rows.append({'Category': 'Hate (binary)', 'Paper train': None, 'HF train': data_metrics['hate_counts']['train'],
             'Paper test': None, 'HF test': data_metrics['hate_counts']['test'],
             'Paper total': PAPER_HATE_TOTAL, 'HF total': hf_hate_total})
counts_cmp = pd.DataFrame(rows).astype({c: 'Int64' for c in ['Paper train', 'Paper test', 'Paper total']})
display(counts_cmp)

save_table(counts_cmp, 'tab_09_paper_vs_hf_counts', float_format='%.0f')
register_asset('T09', 'tables/tab_09_paper_vs_hf_counts.csv (+ .tex)',
               'Category and hate counts in the BanTH paper vs the public HuggingFace release used here',
               NOTEBOOK_NAME, 'III. Dataset / VII. Limitations',
               'TABLE X. Label counts reported in the BanTH paper compared with the public HuggingFace release used in this work. The paper reports seven categories and does not list Misc.')

,Category,Paper train,HF train,Paper test,HF test,Paper total,HF total
0,Political,958,978,122,117,1200,1206
1,Religious,111,110,14,18,139,150
2,Gender,154,180,20,23,193,227
3,Personal Offense,5572,5752,696,715,6964,7177
4,Abusive/Violence,2312,2335,289,304,2888,2932
5,Origin,171,178,22,23,214,227
6,Body Shaming,68,92,10,14,86,123
7,Misc,<NA>,207,<NA>,30,<NA>,262
8,Hate (binary),<NA>,8495,<NA>,1062,10282,10619


Saved: tab_09_paper_vs_hf_counts (.csv and .tex)
Registered T09 -> tables/tab_09_paper_vs_hf_counts.csv (+ .tex)


## Save the baseline details (M04)

The source, the metric definitions, the paper's training settings and the best models are saved to metrics_baseline.json. The training notebook uses this file to compute our metrics the same way as the paper.

In [9]:
baseline = {
    'step': 'baseline',
    'notebook': NOTEBOOK_NAME,
    'date': datetime.now().isoformat(timespec='seconds'),
    'source': {
        'paper': 'BanTH: A Multi-label Hate Speech Detection Dataset for Transliterated Bangla',
        'authors': 'Haider, Shifat, Ishmam, Sourove, Barua, Fahim, Bhuiyan',
        'venue': 'Findings of the Association for Computational Linguistics: NAACL 2025, pp. 7232-7251',
        'doi': '10.18653/v1/2025.findings-naacl.403',
        'table': 'Table 7 (test split), checked against arXiv 2410.13281v4',
    },
    'units': 'percent',
    'metric_definitions': {
        'Binary Macro-F1': 'F1 averaged over the hate and non-hate classes',
        'Binary Macro-Acc.': 'accuracy averaged over the two classes (balanced accuracy)',
        'Multi-label Macro-F1': 'F1 computed per category, then averaged',
        'Subset Acc.': 'share of comments whose full predicted label set is exactly right',
        'Hamming Loss': 'share of individual labels predicted wrongly (lower is better)',
    },
    'paper_finetune_settings': {'learning_rate': 2e-5, 'optimizer': 'Adam', 'max_length': 512,
                                'epochs': 5, 'batch_size': 32,
                                'early_stopping': 'validation loss'},
    'notes': [
        'The paper describes 7 hate categories; the HuggingFace release has 8 (adds Misc).',
        'Paper Table 1 category counts differ from the HuggingFace release (see tab_09).',
        'The paper runs multi-label classification on predicted hate comments; which comments the multi-label scores are computed on is not stated explicitly.',
        'Model sizes (base or large) are not stated in the paper.',
    ],
    'best': best,
    'results': full.to_dict(orient='records'),
}
path = os.path.join(REP_DIR, 'metrics_baseline.json')
with open(path, 'w', encoding='utf-8') as f:
    json.dump(baseline, f, indent=2)
print('Saved:', path)
register_asset('M04', 'evaluation_reports/metrics_baseline.json',
               'Published BanTH baseline results, metric definitions, paper training settings, best model per metric',
               NOTEBOOK_NAME, 'IV. Methodology / V. Results',
               'Not a figure; source for every baseline number quoted in the text.')

Saved: /content/drive/MyDrive/ML_Project/results/evaluation_reports/metrics_baseline.json
Registered M04 -> evaluation_reports/metrics_baseline.json


## Summary

We saved the published BanTH results as our baseline: the full table (T07), a short table for the paper (T08), a comparison of the paper's label counts with the HuggingFace data (T09), and metrics_baseline.json. The best published binary macro-F1 is 77.36 (TB-mBERT), and the best multi-label macro-F1 is 39.53 (GPT 4o + Few-shot), or 30.17 (TB-BERT) among fine-tuned models.

Next, the training notebook reports the same five metrics on the same test split and adds our row to T08.

## Remember this

- What we did: saved the BanTH paper's published test results as our baseline instead of re-training their models.
- Why: it is the same dataset and test split, so we can compare our model with theirs directly, as long as we use the same metrics.
- Key number: the best published binary macro-F1 is 77.36 (TB-mBERT) and the best multi-label macro-F1 is 39.53 (GPT 4o few-shot).